In [40]:
from pathlib import Path

import kagglehub
import pandas as pd

In [ ]:
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").is_dir() and (p / "notebooks").is_dir()),
    Path.cwd(),
)


RAW_DIR = ROOT / "data" / "raw" 
RAW_DIR.mkdir(parents=True, exist_ok=True)

### UNSW-NB15

In [ ]:
OUTPUT = RAW_DIR / "UNSW-NB15.csv"
TEMP = RAW_DIR / "UNSW-NB15.tmp.csv"

In [42]:
download_dir = Path(
    kagglehub.dataset_download("mrwellsdavid/unsw-nb15")
)

files = list(download_dir.rglob("*.csv"))
files = sorted(files, key=lambda f: f.name)
files = files[:5]  


In [43]:
features = pd.read_csv(files[0],encoding="cp1252")
columns = features["Name"].tolist()

total_rows = 0

# Ghi file tạm; chỉ thay merged.csv khi gộp thành công
with TEMP.open("w", encoding="utf-8", newline="") as output:
    for file in files[1:5]:
        file_rows = 0

        for chunk in pd.read_csv(
            file,
            header=None,
            dtype=str,
            keep_default_na=False,
            chunksize=100_000,
        ):
            if chunk.shape[1] != len(columns):
                raise ValueError(
                    f"{file.name}: có {chunk.shape[1]} cột"
                )

            chunk.columns = columns
            chunk.to_csv(output, index=False, header=(total_rows == 0))

            file_rows += len(chunk)
            total_rows += len(chunk)

        print(f"Đã gộp {file.name}: {file_rows:,} dòng")

TEMP.replace(OUTPUT)

print(f"\nTổng cộng: {total_rows:,} dòng, {len(columns)} cột")
print(f"Đã lưu: {OUTPUT}")

Đã gộp UNSW-NB15_1.csv: 700,001 dòng
Đã gộp UNSW-NB15_2.csv: 700,001 dòng
Đã gộp UNSW-NB15_3.csv: 700,001 dòng
Đã gộp UNSW-NB15_4.csv: 440,044 dòng

Tổng cộng: 2,540,047 dòng, 49 cột
Đã lưu: /Users/thonph/Desktop/KLTN/data/raw/UNSW-NB15.csv


## CICIDS-2017

In [45]:
OUTPUT = RAW_DIR / "CICIDS2017.csv"
TEMP = RAW_DIR / "CICIDS2017.tmp.csv"

In [51]:
download_dir = Path(
    kagglehub.dataset_download("chethuhn/network-intrusion-dataset"))

files = list(download_dir.rglob("*.csv"))
files = sorted(files, key=lambda f: f.name)

In [52]:
total_rows = 0
expected_columns = None


with TEMP.open("w", encoding="utf-8", newline="") as output:
    for file in files:
        file_rows = 0

        for chunk in pd.read_csv(
            file,
            header=0,
            dtype=str,
            keep_default_na=False,
            chunksize=100_000,
            encoding="cp1252",
        ):
            # CICIDS2017 có khoảng trắng thừa trong tên cột
            chunk.columns = chunk.columns.str.strip()

            if chunk.columns.duplicated().any():
                raise ValueError(f"{file.name}: có tên cột bị trùng")

            if expected_columns is None:
                expected_columns = chunk.columns.tolist()
            elif set(chunk.columns) != set(expected_columns):
                raise ValueError(
                    f"{file.name}: cấu trúc cột khác các file trước"
                )

            # Đảm bảo các file ghi cùng thứ tự cột
            chunk = chunk[expected_columns]

            chunk.to_csv(
                output,
                index=False,
                header=(total_rows == 0),
            )

            file_rows += len(chunk)
            total_rows += len(chunk)

        print(f"Đã gộp {file.name}: {file_rows:,} dòng")

if total_rows == 0:
    raise ValueError("Không có dòng dữ liệu nào để gộp")

# Chỉ thay file tổng khi gộp hoàn tất
TEMP.replace(OUTPUT)

print(f"\nTổng cộng: {total_rows:,} dòng")
print(f"Số cột: {len(expected_columns)}")
print(f"Đã lưu: {OUTPUT}")

Đã gộp Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 dòng
Đã gộp Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 dòng
Đã gộp Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 dòng
Đã gộp Monday-WorkingHours.pcap_ISCX.csv: 529,918 dòng
Đã gộp Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 dòng
Đã gộp Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 dòng
Đã gộp Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 dòng
Đã gộp Wednesday-workingHours.pcap_ISCX.csv: 692,703 dòng

Tổng cộng: 2,830,743 dòng
Số cột: 79
Đã lưu: /Users/thonph/Desktop/KLTN/data/raw/CICIDS2017.csv
